<a href="https://colab.research.google.com/github/WinsalotNot/HAHA_v1/blob/master/Assignment_2_Multiclass_Perceptron_Andrew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IMPORTS**

In [2]:
import numpy as np
import pandas as pd

# **DATA ORGANIZATION**

In [3]:
# Stores data in data_file (does not include header!)
data_file = pd.read_csv('https://gist.githubusercontent.com/netj/8836201/raw/6f9306ad21398ea43cba4f7d537619d0e07d5ae3/iris.csv')
print(data_file)

# Get the unqiue values in variety for classification
unique_variety = data_file['variety'].unique()
print(unique_variety)

# Take 80% of the dataset RANDOMLY as training data, random_state is the random seed used to ensure reproducibility
training_data_80 = data_file.sample(frac=0.8, random_state=25)
# Take 20% of the dataset by ELIMINATING the training data
testing_data_20 = data_file.drop(training_data_80.index)
print(f'Training Data 1: {(len(training_data_80)/len(data_file)*100)}% Testing Data 1: {(len(testing_data_20)/len(data_file)*100)}%')

# Take 70% of the dataset RANDOMLY as training data, random_state is the random seed used to ensure reproducibility
training_data_70 = data_file.sample(frac=0.7, random_state=24)
# Take 30% of the dataset by ELIMINATING the training data
testing_data_30 = data_file.drop(training_data_70.index)
print(f'Training Data 2: {(len(training_data_70)/len(data_file)*100)}% Testing Data 2: {(len(testing_data_30)/len(data_file)*100)}%')

# Take 60% of the dataset RANDOMLY as training data, random_state is the random seed used to ensure reproducibility
training_data_60 = data_file.sample(frac=0.6, random_state=23)
# Take 40% of the dataset by ELIMINATING the training data
testing_data_40 = data_file.drop(training_data_60.index)
print(f'Training Data 3: {(len(training_data_60)/len(data_file)*100)}% Testing Data 3: {(len(testing_data_40)/len(data_file)*100)}%')


     sepal.length  sepal.width  petal.length  petal.width    variety
0             5.1          3.5           1.4          0.2     Setosa
1             4.9          3.0           1.4          0.2     Setosa
2             4.7          3.2           1.3          0.2     Setosa
3             4.6          3.1           1.5          0.2     Setosa
4             5.0          3.6           1.4          0.2     Setosa
..            ...          ...           ...          ...        ...
145           6.7          3.0           5.2          2.3  Virginica
146           6.3          2.5           5.0          1.9  Virginica
147           6.5          3.0           5.2          2.0  Virginica
148           6.2          3.4           5.4          2.3  Virginica
149           5.9          3.0           5.1          1.8  Virginica

[150 rows x 5 columns]
['Setosa' 'Versicolor' 'Virginica']
Training Data 1: 80.0% Testing Data 1: 20.0%
Training Data 2: 70.0% Testing Data 2: 30.0%
Training Data 3: 60.0%

In [4]:
def compute_confusion_matrix(actual_labels, predicted_labels, num_classes):
    # Initialize a num_classes x num_classes matrix with zeros
    confusion_matrix = [[0] * num_classes for _ in range(num_classes)]

    # Fill the confusion matrix
    for actual, predicted in zip(actual_labels, predicted_labels):
        confusion_matrix[actual][predicted] += 1

    return confusion_matrix

# **MULTICLASS PERCEPTRON ALGORITHM**

In [5]:
def multiclass_perceptron(data, weight_2D_array, learning_rate, epoch, isTesting):
  # Takes all row and all except last [:, :-1], then converts to numpy compatible array
  data_numpy = data.iloc[:, :-1].to_numpy()

  # Each unqiue data in 'variety' are represented sequentially from 0
  result_each_indexes = data.iloc[:, -1].to_numpy()                                         # Takes all rows and ONLY the last column [:, -1], then converts to numpy compatible array
  unique_labels, results_given_int = np.unique(result_each_indexes, return_inverse=True)    # Gets an array of the labels of each uniques and an array of the unique values mapped to their respective indexes
  label_to_index = {label: idx for idx, label in enumerate(unique_labels)}                  # Shows which unqiue values are assigned to which index
  print("Label Mapping:", label_to_index)
  print("Converted Indexes:", results_given_int)

  updated_weights = weight_2D_array.copy()
  best_weights = updated_weights.copy()
  best_accuracy = 0.0
  best_epoch = 0
  confusion_matrix = []

  if isTesting:
    weighted_sums_testing = np.dot(data_numpy, updated_weights.T)
    results_calculated_testing = np.argmax(weighted_sums_testing, axis=1)
    correct_predictions_testing = np.sum(results_calculated_testing == results_given_int)
    accuracy_testing = (correct_predictions_testing / len(data_numpy) * 100)
    confusion_matrix = compute_confusion_matrix(results_given_int, results_calculated_testing, len(unique_labels))

    print(f'Testing Accuracy: {accuracy_testing}%')
    print(f'Testing Results: {results_calculated_testing}')
    print(f'Compared To: {results_given_int}')

    return accuracy_testing, confusion_matrix


  else:
    iteration = 0
    success = False
    while (iteration < epoch):
      weighted_sums = np.dot(data_numpy, updated_weights.T) # Uses dot matrix calculation as well as tansposing the updated_weights (turning the features into rows and classes in columns)
      results_calculated = np.argmax(weighted_sums, axis=1) # Because classes are now rows, take the biggest one from each row

      correct_predictions = np.sum(results_calculated == results_given_int)
      accuracy = correct_predictions / len(data_numpy)  # Compute accuracy

      # Track best accuracy and corresponding weights
      if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_weights = updated_weights.copy()
        best_epoch = iteration + 1
        confusion_matrix = compute_confusion_matrix(results_given_int, results_calculated, len(unique_labels))


      print(f'Epoch {iteration + 1}: {correct_predictions}/{len(data_numpy)} samples correctly classified.')

      if correct_predictions == len(data_numpy):  # If 100% correct, terminate early
        print(f'Converged at epoch {iteration + 1}')
        success = True
        break

      for index, (result_calculated, result_given_int) in enumerate(zip(results_calculated, results_given_int)):
        if result_calculated != result_given_int:
            updated_weights[result_given_int] += learning_rate * data_numpy[index]  # Increases the expected class
            updated_weights[result_calculated] -= learning_rate * data_numpy[index] # Decreases the predicted class

      iteration += 1

    if not success:
      print(f'Given {epoch} epoch, weights did not converge: {updated_weights}')

    print(f'Best accuracy: {best_accuracy * 100:.2f}%')
    print(f'Best weights:\n{best_weights}')
    print(f'Best weight achieved at epoch {best_epoch}')
    return best_weights, (best_accuracy * 100), best_epoch, confusion_matrix


In [18]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(0)

# Define parameters
num_features = 4
num_classes = 3

# Initialize weights
weight_2D_range = np.random.uniform(low=-0.5, high=0.5, size=(num_classes, num_features))
weight_2D_zeros = np.zeros((num_classes, num_features))

def display_results(results):
    df = pd.DataFrame(results, columns=["Split", "Init Weights", "Learning Rate", "Train Accuracy", "Epochs", "Test Accuracy"])
    print(df.to_string(index=False))

# Store experiment results
experiment_results = []
confusion_matrices = []

# Define data splits
splits = [("80/20", training_data_80, testing_data_20),
          ("70/30", training_data_70, testing_data_30),
          ("60/40", training_data_60, testing_data_40)]

# Define learning rates and initial weight configurations
learning_rates = [0.1, 0.01]
initial_weights = [("Zeros", weight_2D_zeros), ("Range", weight_2D_range)]

# Run experiments
for split_name, train_data, test_data in splits:
    for weight_name, weight_matrix in initial_weights:
        for lr in learning_rates:
            # Train perceptron
            trained_weights, training_accuracy, epochs, confusion_matrix_training = multiclass_perceptron(train_data, weight_matrix, lr, 1000, False)

            # Test perceptron and get predictions
            testing_accuracy, confusion_matrix_testing = multiclass_perceptron(test_data, trained_weights, None, None, True)

            # Store results
            experiment_results.append([split_name, weight_name, lr, training_accuracy, epochs, testing_accuracy])
            confusion_matrices.append((split_name, weight_name, lr, confusion_matrix_training))

# Display experiment results
display_results(experiment_results)

def display_confusion_matrix(conf_matrix):
    num_classes = len(conf_matrix)

    # Create row and column labels
    headers = [f"Pred {i}" for i in range(num_classes)]
    index_labels = [f"Actual {i}" for i in range(num_classes)]

    # Convert to DataFrame
    df = pd.DataFrame(conf_matrix, index=index_labels, columns=headers)

    # Print nicely formatted table
    print("\nConfusion Matrix:")
    print(df.to_string(index=True))

# Display confusion matrices
print("\nConfusion Matrices:")
for split_name, weight_name, lr, conf_matrix in confusion_matrices:
    print(f"\nSplit: {split_name}, Init Weights: {weight_name}, Learning Rate: {lr}")
    display_confusion_matrix(confusion_matrix_training)


Streaming output truncated to the last 5000 lines.
Epoch 213: 102/105 samples correctly classified.
Epoch 214: 100/105 samples correctly classified.
Epoch 215: 101/105 samples correctly classified.
Epoch 216: 98/105 samples correctly classified.
Epoch 217: 99/105 samples correctly classified.
Epoch 218: 98/105 samples correctly classified.
Epoch 219: 101/105 samples correctly classified.
Epoch 220: 103/105 samples correctly classified.
Epoch 221: 103/105 samples correctly classified.
Epoch 222: 103/105 samples correctly classified.
Epoch 223: 103/105 samples correctly classified.
Epoch 224: 103/105 samples correctly classified.
Epoch 225: 103/105 samples correctly classified.
Epoch 226: 103/105 samples correctly classified.
Epoch 227: 103/105 samples correctly classified.
Epoch 228: 103/105 samples correctly classified.
Epoch 229: 103/105 samples correctly classified.
Epoch 230: 103/105 samples correctly classified.
Epoch 231: 103/105 samples correctly classified.
Epoch 232: 102/105 sa